# The Buffalo Times

Shruti Priya (50757311) · EAS 503 extra credit
[Live app](https://newsappeas503-ihbxry75s9tguguaepuerb.streamlit.app) · [Code](https://github.com/shruti23-ui/Newsapp_eas503)

Three missed opportunities in the class news app, each turned into an analysis:

| # | Missed opportunity | Feature |
|---|---|---|
| 1 | spaCy entities were computed but never used | Entity Explorer |
| 2 | Each class notebook saved a different day's news, never compared | Echo Index |
| 3 | Gemini's JSON summary was never parsed or checked | Newsroom Check |

Extensions: Gemini model comparison, news framing, daily snapshots (GitHub Actions). The live steps (Tavily fact-check,
Gemini rewrite, ElevenLabs audio, Ask) run in the app. This notebook reproduces the analysis offline on the four
NewsAPI snapshots saved in the class notebooks.

## Setup
```bash
pip install -r requirements.txt
```

In [1]:
import os
os.environ["DEMO_MODE"] = "true"
os.environ["ENABLE_ZSC"] = "false"

import pandas as pd
import plotly.express as px
import plotly.io as pio
from scipy.stats import spearmanr

from buffalo import ai_summary, analytics, config, data_pipeline, nlp

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 90)

## Data

In [2]:
df, _ = data_pipeline.get_news_data()
df, _ = nlp.enrich(df)
print(df["uid"].nunique(), "unique headlines")
df.pivot_table(index="snapshot", columns="category", values="uid", aggfunc="nunique")

490 unique headlines


category,business,entertainment,general,health,science,sports,technology
snapshot,,,,,,,
2026-10-19,19,20,17,19,19,19,18
2026-10-21,20,18,19,19,19,20,20
2026-10-26,16,20,18,18,19,20,18
2026-11-02,18,19,19,18,20,20,20


NewsAPI's "top" order compared with publish time, per snapshot and category (−1 = newest first):

In [3]:
rho = [spearmanr(range(len(g)), g["publishedAt"].astype("int64")).statistic
       for _, g in df.dropna(subset=["publishedAt"]).groupby(["snapshot", "category"]) if len(g) > 2]
print(len(rho), "groups, mean Spearman", round(sum(rho) / len(rho), 2))

28 groups, mean Spearman -1.0


## 1 · Entity Explorer
spaCy entities, with an EntityRuler correcting names the small model mislabels.

In [4]:
E = analytics.entity_rows(df)
labels = dict(E.drop_duplicates("entity")[["entity", "label"]].values)
{name: labels.get(name) for name in ["Donald Trump", "Nvidia", "AI"]}

{'Donald Trump': 'PERSON', 'Nvidia': 'ORG', 'AI': None}

In [5]:
top = analytics.entity_summary(E).head(15)
fig = px.bar(top, x="mentions", y="entity", orientation="h", color="mean_polarity",
             color_continuous_scale="RdBu", range_color=[-0.5, 0.5], title="Most mentioned entities")
fig.update_yaxes(autorange="reversed")
fig.show()
top.round(3)

,entity,mentions,label,mean_polarity,snapshots
292,NFL,16,ORG,0.151,4
105,Donald Trump,12,PERSON,0.061,4
435,United States,12,GPE,-0.003,4
113,Earth,7,LOC,-0.008,4
397,SpaceX,7,ORG,0.069,4
22,Anthropic,7,ORG,-0.051,3
23,Apple,6,ORG,0.058,3
464,Xbox,6,ORG,0.143,2
299,Netflix,5,ORG,0.086,4
275,Microsoft,5,ORG,-0.020,3


In [6]:
names = list(top["entity"])
(E[E["entity"].isin(names)].drop_duplicates(["uid", "entity"])
   .pivot_table(index="entity", columns="category", values="uid", aggfunc="count", fill_value=0)
   .reindex(names))

category,business,entertainment,general,health,science,sports,technology
entity,,,,,,,
NFL,0,0,0,0,0,16,0
Donald Trump,3,2,7,0,0,0,0
United States,2,1,3,3,1,1,1
Earth,0,0,0,0,7,0,0
SpaceX,5,0,0,0,2,0,0
Anthropic,4,1,1,0,0,0,1
Apple,1,0,0,0,0,0,5
Xbox,0,0,0,0,0,0,6
Netflix,2,3,0,0,0,0,0


In [7]:
pairs = analytics.cooccurrence(E, names)
pd.DataFrame([(a, b, n) for (a, b), n in pairs.most_common(8)], columns=["entity A", "entity B", "headlines together"])

,entity A,entity B,headlines together
0,Iran,United States,3
1,Donald Trump,Iran,2
2,Donald Trump,United States,2
3,SpaceX,United States,1
4,NFL,Patriots,1
5,Google,Microsoft,1
6,Microsoft,Xbox,1
7,Anthropic,Donald Trump,1


## 2 · Echo Index
Echo = highest TF-IDF cosine similarity of a headline to any headline from an earlier snapshot.

In [8]:
echo = analytics.echo_scores(df)
trend = echo.groupby("snapshot")["echo"].mean()
px.line(trend.reset_index(), x="snapshot", y="echo", markers=True, title="Average echo per snapshot").show()
trend.round(3)

snapshot
2026-10-19      NaN
2026-10-21    0.118
2026-10-26    0.163
2026-11-02    0.186
Name: echo, dtype: float64

In [9]:
echo.sort_values("echo", ascending=False)[["snapshot", "title", "echo", "echo_of"]].head(5).round(2)

,snapshot,title,echo,echo_of
444,2026-11-02,"Game ball, three stars, and snap count analysis: Week 8, Eagles at Commanders",0.81,"Game ball, three stars, and snap count analysis: Week 6, Eagles vs. Panthers"
272,2026-10-26,"Horoscope for Monday, October 26, 2026",0.74,"Horoscope for Monday, October 19, 2026"
451,2026-11-02,5 winners and losers from Patriots’ win over the Dolphins,0.74,6 winners and losers from Patriots’ win over Jets
286,2026-10-26,How hot you drink your tea may affect heart health,0.53,How you drink your tea could determine whether you gain or lose heart health benefits
320,2026-10-26,"NFL Week 7 winners, losers led by Jets' Aaron Glenn running out of chances",0.53,The Winners and Losers of the NFL Week 6


Rising and fading words: log-odds with an informative Dirichlet prior (Monroe, Colaresi & Quinn, 2008).

In [10]:
terms = analytics.term_shift(df, df["snapshot"].max())
print("largest |z|:", round(terms["z"].abs().max(), 2))
pd.concat([terms.head(6), terms.tail(6)]).round(2)

largest |z|: 1.69


,word,z,focus_count,earlier_count
0,jack,1.69,3,0
1,billion,1.48,5,3
2,calls,1.38,3,1
3,defense,1.38,3,1
4,war,1.38,3,1
5,midterm,1.38,3,1
269,secret,-0.91,0,6
270,oct,-0.91,0,6
271,iphone,-0.91,0,6
272,best,-0.98,0,7


## 3 · Newsroom Check
Coverage of the saved Nov 2 Gemini summary: a headline is covered when its similarity to a summary cluster is at least 0.10.

In [11]:
summary = ai_summary.load_summaries()["2026-11-02"]
audit = analytics.audit_summary(df[df["snapshot"] == "2026-11-02"], summary)
cov = analytics.coverage_by_category(audit["heads"], 0.10, config.CATEGORIES)
print(f"covered: {cov['represented'].sum() / cov['total'].sum():.0%} |",
      "left out:", list(cov.loc[cov["represented"] == 0, "category"]))
px.bar(cov, x="share", y="category", orientation="h", range_x=[0, 1], title="Coverage by category").show()

covered: 11% | left out: ['entertainment', 'science', 'technology']


In [12]:
pd.DataFrame([{"threshold": th, "coverage": round(c["represented"].sum() / c["total"].sum(), 3),
               "categories left out": int((c["represented"] == 0).sum())}
              for th in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
              for c in [analytics.coverage_by_category(audit["heads"], th, config.CATEGORIES)]])

,threshold,coverage,categories left out
0,0.05,0.216,1
1,0.10,0.112,3
2,0.15,0.080,4
3,0.20,0.072,4
4,0.25,0.040,4
5,0.30,0.024,4


In [13]:
audit["sents"].sort_values("support")[["where", "sentence", "support", "best_headline"]].head(5).round(2)

,where,sentence,support,best_headline
0,Geopolitics and International Conflict,"Concurrently, regional security reports highlight ongoing challenges and cyber inciden...",0.08,Washington reports first US bird flu death of 2026; health officials say risk to the p...
1,Global summary,"In the sports arena, fans are processing major NFL injuries and weekend game takeaways...",0.17,"NFL trade deadline live updates: Latest news, rumors after Week 8"
2,Health and Disease Outbreaks,"Additionally, new studies continue to shed light on various medical conditions, rangin...",0.20,"Energy drinks linked to higher risk of cardiac arrest in young adults, study finds"
3,Global summary,Today's news spans dynamic financial markets reacting to midterm elections and corpora...,0.23,2026 Midterm Elections Live Updates
4,Stock Market and Corporate Financials,"Meanwhile, several major tech and finance companies are making moves, including Anthro...",0.32,"Oracle files to sell $20 billion in new stock, bonds to finance AI buildout"


## Extension · Gemini model comparison
The same prompt on two models, two runs per model per day for 8 days, each summary scored with the same audit.

In [14]:
runs = pd.read_csv(config.MODEL_RUNS_FILE)
model_summary, per_day, test = analytics.compare_models(runs)
print(f"{test['a']} vs {test['b']}: {-test['mean_diff'] * 100:+.1f} points coverage, "
      f"Wilcoxon signed-rank p = {test['p']:.3f} (n = {test['n']} days)")
px.line(per_day.reset_index().melt(id_vars="snapshot", var_name="model", value_name="coverage"),
        x="snapshot", y="coverage", color="model", markers=True, title="Coverage per day").show()
model_summary.round(3)

gemini-3.5-flash vs gemini-3.5-flash-lite: +6.2 points coverage, Wilcoxon signed-rank p = 0.008 (n = 8 days)


,model,runs,valid_json,coverage,coverage_sd,left_out,weak_share
0,gemini-3.5-flash,15,1.0,0.244,0.095,1.8,0.516
1,gemini-3.5-flash-lite,16,1.0,0.173,0.098,3.0,0.638


## Extension · News framing
Zero-shot frames (`facebook/bart-large-mnli`) from Semetko & Valkenburg (2000).

In [15]:
frames = pd.read_csv(config.FRAMING_DIR / "frames.csv")
class_frames = frames[frames["uid"].isin(set(df["uid"]))]
mix = analytics.frame_mix(class_frames, "category")
px.bar(mix.reset_index().melt(id_vars="category", var_name="frame", value_name="share"),
       x="share", y="category", color="frame", orientation="h", title="Frames by category").show()
print(f"model unsure (confidence under 40%): {(class_frames['score'] < 0.4).mean():.0%}")
class_frames["frame"].value_counts(normalize=True).round(3)

model unsure (confidence under 40%): 15%


frame
conflict          0.539
human interest    0.273
economic          0.161
responsibility    0.027
Name: proportion, dtype: float64

In [16]:
class_frames.sort_values("score", ascending=False).groupby("frame").head(2)[["frame", "title", "category", "score"]]

,frame,title,category,score
168,human interest,How a photo of a person’s face could help predict whether they’ll survive cancer treat...,health,0.964
405,human interest,"At 58, I Had Early Signs of Alzheimer’s in My Brain. These Interventions Got Rid of Them",health,0.930
342,conflict,Conflicting reports about T-Mobile's special service in parts of the US,technology,0.921
141,economic,Anthropic posts $80bn loss in first half of 2026: WSJ,business,0.905
61,conflict,"Scientists say Earth only has six continents, not seven",science,0.886
11,economic,Robot factories and hypersonic missiles: JPMorgan Chase’s first $5 billion of defense ...,business,0.881
445,responsibility,The Kansas City Chiefs are searching for answers,sports,0.543
90,responsibility,The Winners and Losers of the NFL Week 6,sports,0.541


## Extension · Opinionated wording
TextBlob subjectivity by category (0 = factual, 1 = opinion).

In [17]:
(analytics.unique_articles(df).groupby("category")
   .agg(headlines=("uid", "count"), mean_polarity=("polarity", "mean"), mean_subjectivity=("subjectivity", "mean"),
        share_factual=("subjectivity", lambda s: (s < 0.5).mean()))
   .sort_values("mean_subjectivity", ascending=False).round(3))

,headlines,mean_polarity,mean_subjectivity,share_factual
category,,,,
technology,76,0.079,0.388,0.645
entertainment,77,-0.036,0.327,0.688
sports,79,0.138,0.312,0.684
general,34,0.010,0.300,0.706
health,74,0.062,0.295,0.743
science,77,0.020,0.287,0.701
business,73,0.081,0.258,0.753
